In [1]:
# Pipeline LightGBM Optimizado para Codespaces - Sin SMOTE
# Objetivo: Mejorar P=0.047 → 0.10+, mantener R~0.70

import warnings, os, gc, time, json, joblib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import datetime

from sklearn.metrics import (
    precision_score, recall_score, f1_score, fbeta_score, roc_auc_score,
    precision_recall_curve, average_precision_score, confusion_matrix,
    classification_report, make_scorer
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif

from imblearn.under_sampling import RandomUnderSampler, EditedNearestNeighbours

try:
    from lightgbm import LGBMClassifier
    import lightgbm as lgb
    print("✅ LightGBM disponible")
except ImportError:
    print("❌ LightGBM no disponible")
    exit(1)

✅ LightGBM disponible


In [ ]:


# ==========================
# CONFIG OPTIMIZADA PARA CODESPACES
# ==========================
DATA_PATH = "../data/processed/df_ready_model.csv"
SAVE_DIR = "../models"
RESULTS_DIR = "../results"
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

RANDOM_STATE = 42
TARGET_RECALL = 0.70      
TARGET_PRECISION = 0.10   
VAL_YEAR = 2023
VAL_MONTH = 12
TEST_YEAR = 2024

# Reducir features para memoria
MAX_FEATURES = 20  

print("🚀 PIPELINE OPTIMIZADO PARA CODESPACES")
print("=" * 60)
print(f"🎯 Objetivo: P≥{TARGET_PRECISION:.0%}, R≥{TARGET_RECALL:.0%}")
print(f"💾 Modo: Eficiente en memoria, sin SMOTE")

def log_progress(message):
    timestamp = datetime.now().strftime("%H:%M:%S")
    print(f"[{timestamp}] {message}")

# ==========================
# FUNCIONES EFICIENTES EN MEMORIA
# ==========================
def create_cluster_features_efficient(df_train, df_apply, sample_size=100000):
    """Features de cluster eficientes en memoria"""
    
    log_progress("Creando features de cluster (modo eficiente)...")
    
    # Samplear si train es muy grande
    if len(df_train) > sample_size:
        df_train_sample = df_train.sample(n=sample_size, random_state=RANDOM_STATE)
        log_progress(f"  Usando muestra de {sample_size:,} registros para estadísticas")
    else:
        df_train_sample = df_train
    
    # Columnas meteorológicas con valores por defecto
    weather_cols = {
        "temperature_2m (°C)": 15.0,
        "precipitation (mm)": 0.0,
        "wind_speed_10m (km/h)": 10.0
    }
    
    for col, default_val in weather_cols.items():
        if col not in df_train.columns:
            df_train[col] = default_val
            df_apply[col] = default_val
    
    # 1. Estadísticas básicas por cluster (solo las más importantes)
    cluster_stats = df_train_sample.groupby("cluster_id").agg({
        "accident": ["mean", "sum", "count"],
        list(weather_cols.keys())[0]: ["mean", "std"],
        list(weather_cols.keys())[1]: ["mean", "max"],
        list(weather_cols.keys())[2]: ["mean", "max"]
    }).round(4)
    
    cluster_stats.columns = ['_'.join(col).strip() for col in cluster_stats.columns.values]
    cluster_stats = cluster_stats.add_prefix("cluster_")
    
    # 2. Patrones temporales simples
    # Por hora (solo las más relevantes)
    hour_stats = df_train_sample.groupby(["cluster_id", "hour"])["accident"].mean().reset_index()
    hour_stats.columns = ["cluster_id", "hour", "cluster_hour_rate"]
    
    # Por día de semana
    dow_stats = df_train_sample.groupby(["cluster_id", "day_of_week"])["accident"].mean().reset_index()
    dow_stats.columns = ["cluster_id", "day_of_week", "cluster_dow_rate"]
    
    # 3. Risk scoring simple
    risk_scores = df_train_sample.groupby("cluster_id")["accident"].mean().reset_index()
    risk_scores.columns = ["cluster_id", "cluster_risk_score"]
    risk_scores["cluster_risk_level"] = pd.qcut(
        risk_scores["cluster_risk_score"], 
        q=3, 
        labels=[0, 1, 2],  # Bajo, Medio, Alto como numérico
        duplicates='drop'
    ).astype(int)
    
    # Merge eficiente
    result = df_apply.copy()
    
    # Usar merge con validate para detectar problemas
    result = pd.merge(result, cluster_stats, on="cluster_id", how="left", validate="many_to_one")
    result = pd.merge(result, hour_stats, on=["cluster_id", "hour"], how="left", validate="many_to_one")
    result = pd.merge(result, dow_stats, on=["cluster_id", "day_of_week"], how="left", validate="many_to_one")
    result = pd.merge(result, risk_scores, on="cluster_id", how="left", validate="many_to_one")
    
    # Rellenar NaN con medianas
    numeric_cols = result.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if col.startswith("cluster_") and result[col].isna().any():
            result[col] = result[col].fillna(result[col].median())
    
    # Limpiar memoria
    del df_train_sample, cluster_stats, hour_stats, dow_stats, risk_scores
    gc.collect()
    
    log_progress(f"✅ Features de cluster creadas: {len([c for c in result.columns if c.startswith('cluster_')])}")
    
    return result

def create_temporal_features_simple(df):
    """Features temporales básicas"""
    
    # Cíclicas principales
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
    df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
    
    # Indicadores binarios
    df["is_rush_hour"] = df["hour"].isin([7,8,9,17,18,19]).astype(int)
    df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 5)).astype(int)
    
    # Interacciones simples
    df["weekend_night"] = df["is_weekend"] * df["is_night"]
    df["weekday_rush"] = (1 - df["is_weekend"]) * df["is_rush_hour"]
    
    return df

def create_weather_features_simple(df):
    """Features meteorológicas simples"""
    
    # Solo si existen las columnas
    if "temperature_2m (°C)" in df.columns:
        df["temp_cold"] = (df["temperature_2m (°C)"] < 10).astype(int)
        df["temp_hot"] = (df["temperature_2m (°C)"] > 25).astype(int)
    
    if "precipitation (mm)" in df.columns:
        df["has_rain"] = (df["precipitation (mm)"] > 0.1).astype(int)
        df["heavy_rain"] = (df["precipitation (mm)"] > 5).astype(int)
    
    if "wind_speed_10m (km/h)" in df.columns:
        df["high_wind"] = (df["wind_speed_10m (km/h)"] > 20).astype(int)
    
    return df

def custom_fbeta_scorer(beta=0.5):
    """F-beta score con beta < 1 para priorizar precision"""
    return make_scorer(fbeta_score, beta=beta)

# ==========================
# PIPELINE PRINCIPAL
# ==========================
start_time = time.time()

log_progress("Iniciando pipeline...")

# 1. CARGA DE DATOS CON DTYPES OPTIMIZADOS
log_progress("Cargando dataset con tipos optimizados...")

# Definir dtypes para reducir memoria
dtype_dict = {
    'cluster_id': 'int16',
    'accident': 'int8',
    'year': 'int16',
    'month': 'int8',
    'day': 'int8',
    'hour': 'int8',
    'day_of_week': 'int8',
    'is_weekend': 'int8',
    'Fiesta': 'int8',
    'dia_festivo': 'int8',
    'temperature_2m (°C)': 'float32',
    'precipitation (mm)': 'float32',
    'wind_speed_10m (km/h)': 'float32'
}

df = pd.read_csv(DATA_PATH, dtype=dtype_dict, low_memory=False)
log_progress(f"Dataset: {df.shape[0]:,} registros, {df.shape[1]} columnas")
log_progress(f"Memoria usada: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

# Splits temporales
train_mask = df["year"] <= 2022
val_mask = (df["year"] == VAL_YEAR) & (df["month"] == VAL_MONTH)
test_mask = df["year"] == TEST_YEAR

df_train = df.loc[train_mask].copy()
df_val = df.loc[val_mask].copy()
df_test = df.loc[test_mask].copy()

# Liberar memoria inmediatamente
del df
gc.collect()

log_progress(f"Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,}")
log_progress(f"Accidentes - Train: {df_train['accident'].mean()*100:.2f}%")

# 2. FEATURE ENGINEERING EFICIENTE
log_progress("Feature engineering eficiente...")

# Features de cluster (con sampling)
df_train_fe = create_cluster_features_efficient(df_train, df_train, sample_size=200000)
df_val_fe = create_cluster_features_efficient(df_train, df_val, sample_size=200000)
df_test_fe = create_cluster_features_efficient(df_train, df_test, sample_size=200000)

# Liberar df_train original
del df_train
gc.collect()

# Features temporales
df_train_fe = create_temporal_features_simple(df_train_fe)
df_val_fe = create_temporal_features_simple(df_val_fe)
df_test_fe = create_temporal_features_simple(df_test_fe)

# Features meteorológicas
df_train_fe = create_weather_features_simple(df_train_fe)
df_val_fe = create_weather_features_simple(df_val_fe)
df_test_fe = create_weather_features_simple(df_test_fe)

# Identificar features
feature_cols = [col for col in df_train_fe.columns 
               if col not in ["accident", "time_n", "geometry", "year"]]

log_progress(f"Total features: {len(feature_cols)}")

# 3. PREPARAR MATRICES
X_train_full = df_train_fe[feature_cols].values.astype(np.float32)
y_train_full = df_train_fe["accident"].values.astype(np.int8)

X_val = df_val_fe[feature_cols].values.astype(np.float32)
y_val = df_val_fe["accident"].values.astype(np.int8)

X_test = df_test_fe[feature_cols].values.astype(np.float32)
y_test = df_test_fe["accident"].values.astype(np.int8)

# Liberar DataFrames
del df_train_fe, df_val_fe, df_test_fe
gc.collect()

# 4. UNDERSAMPLING INTELIGENTE CON ENN
log_progress("Aplicando undersampling inteligente...")

# Primero RandomUnderSampler para reducir mayoría
rus = RandomUnderSampler(
    sampling_strategy=0.05,  # 5% de positivos
    random_state=RANDOM_STATE
)
X_temp, y_temp = rus.fit_resample(X_train_full, y_train_full)

# Luego EditedNearestNeighbours para limpiar ruido
enn = EditedNearestNeighbours(
    sampling_strategy='majority',
    n_neighbors=3,
    kind_sel='mode'
)
X_balanced, y_balanced = enn.fit_resample(X_temp, y_temp)

log_progress(f"Dataset balanceado: {len(X_balanced):,} registros ({y_balanced.mean()*100:.2f}% accidentes)")

# Liberar memoria
del X_train_full, y_train_full, X_temp, y_temp
gc.collect()

# 5. FEATURE SELECTION RÁPIDA
log_progress(f"Selección de {MAX_FEATURES} mejores features...")

# Solo F-statistic para velocidad
selector = SelectKBest(score_func=f_classif, k=MAX_FEATURES)
X_balanced_selected = selector.fit_transform(X_balanced, y_balanced)
X_val_selected = selector.transform(X_val)
X_test_selected = selector.transform(X_test)

selected_features = [feature_cols[i] for i in selector.get_support(indices=True)]
log_progress(f"Features seleccionadas: {len(selected_features)}")

# 6. NORMALIZACIÓN
log_progress("Normalizando features...")
scaler = StandardScaler()
X_balanced_scaled = scaler.fit_transform(X_balanced_selected).astype(np.float32)
X_val_scaled = scaler.transform(X_val_selected).astype(np.float32)
X_test_scaled = scaler.transform(X_test_selected).astype(np.float32)

# Liberar memoria
del X_balanced, X_balanced_selected, X_val_selected, X_test_selected
gc.collect()

# 7. BÚSQUEDA DE HIPERPARÁMETROS CON RANDOM SEARCH (más rápido)
log_progress("Búsqueda de hiperparámetros con RandomizedSearch...")

param_distributions = {
    "n_estimators": [200, 300, 400, 500],
    "learning_rate": [0.03, 0.05, 0.08, 0.1],
    "num_leaves": [20, 31, 50, 70],
    "min_child_samples": [20, 50, 100],
    "reg_alpha": [0.1, 0.5, 1.0, 2.0],
    "reg_lambda": [0.1, 0.5, 1.0, 2.0],
    "min_split_gain": [0.01, 0.1, 0.5],
    "subsample": [0.7, 0.8, 0.9],
    "colsample_bytree": [0.7, 0.8, 0.9]
}

base_model = LGBMClassifier(
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
    importance_type='gain'
)

# RandomizedSearch es más rápido que GridSearch
cv_strategy = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

rs = RandomizedSearchCV(
    base_model, 
    param_distributions,
    n_iter=30,  # Solo 30 combinaciones
    scoring=custom_fbeta_scorer(beta=0.5),
    cv=cv_strategy,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE
)

rs.fit(X_balanced_scaled, y_balanced)

log_progress(f"✅ Búsqueda completada")
log_progress(f"Mejores parámetros: {rs.best_params_}")

# 8. REENTRENAMIENTO CON EARLY STOPPING
log_progress("Reentrenando con early stopping...")

best_model = LGBMClassifier(
    **rs.best_params_,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

# Early stopping
eval_set = [(X_val_scaled, y_val)]
best_model.fit(
    X_balanced_scaled, y_balanced,
    eval_set=eval_set,
    eval_metric="logloss",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

# 9. CALIBRACIÓN DE PROBABILIDADES
log_progress("Calibrando probabilidades...")

# Usar sigmoid que es más rápido que isotonic
calibrated_model = CalibratedClassifierCV(
    best_model, 
    method='sigmoid',
    cv=3
)
calibrated_model.fit(X_balanced_scaled, y_balanced)

# 10. OPTIMIZACIÓN DE THRESHOLD
log_progress("Optimizando threshold...")

proba_val = calibrated_model.predict_proba(X_val_scaled)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, proba_val)

# Estrategia: Maximizar F-0.5
f_betas = []
for p, r in zip(precisions[:-1], recalls[:-1]):
    if p > 0 and r > 0:
        f_beta = (1 + 0.5**2) * (p * r) / (0.5**2 * p + r)
    else:
        f_beta = 0
    f_betas.append(f_beta)

idx_optimal = np.argmax(f_betas)
optimal_threshold = thresholds[idx_optimal]
val_precision = precisions[idx_optimal]
val_recall = recalls[idx_optimal]

log_progress(f"Threshold óptimo: {optimal_threshold:.4f}")
log_progress(f"Val - P: {val_precision:.3f}, R: {val_recall:.3f}")

# Si recall muy bajo, buscar alternativa
if val_recall < TARGET_RECALL:
    valid_idx = np.where(recalls[:-1] >= TARGET_RECALL)[0]
    if len(valid_idx) > 0:
        idx_alt = valid_idx[np.argmax(precisions[:-1][valid_idx])]
        optimal_threshold = thresholds[idx_alt]
        log_progress(f"Threshold ajustado para recall≥{TARGET_RECALL}: {optimal_threshold:.4f}")

# 11. EVALUACIÓN EN TEST
log_progress("\n📊 EVALUACIÓN FINAL EN TEST")
log_progress("=" * 50)

proba_test = calibrated_model.predict_proba(X_test_scaled)[:, 1]
y_pred_test = (proba_test >= optimal_threshold).astype(int)

# Métricas
test_precision = precision_score(y_test, y_pred_test)
test_recall = recall_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test)
test_f05 = fbeta_score(y_test, y_pred_test, beta=0.5)
test_pr_auc = average_precision_score(y_test, proba_test)
test_roc_auc = roc_auc_score(y_test, proba_test)

cm = confusion_matrix(y_test, y_pred_test)
tn, fp, fn, tp = cm.ravel()

print(f"\n🎯 MÉTRICAS FINALES:")
print(f"   Precision:  {test_precision:.3f} ({test_precision*100:.1f}%)")
print(f"   Recall:     {test_recall:.3f} ({test_recall*100:.1f}%)")
print(f"   F1-Score:   {test_f1:.3f}")
print(f"   F0.5-Score: {test_f05:.3f}")
print(f"   PR-AUC:     {test_pr_auc:.4f}")
print(f"   ROC-AUC:    {test_roc_auc:.4f}")

print(f"\n🔢 MATRIZ DE CONFUSIÓN:")
print(f"   Verdaderos Negativos:  {tn:,}")
print(f"   Falsos Positivos:      {fp:,}")
print(f"   Falsos Negativos:      {fn:,}")
print(f"   Verdaderos Positivos:  {tp:,}")

# Comparación
print(f"\n📈 COMPARACIÓN CON BASELINE:")
baseline_p, baseline_r, baseline_auc = 0.047, 0.737, 0.069
print(f"   Baseline: P={baseline_p:.3f}, R={baseline_r:.3f}, AUC={baseline_auc:.3f}")
print(f"   Actual:   P={test_precision:.3f}, R={test_recall:.3f}, AUC={test_pr_auc:.3f}")

improvement_p = (test_precision / baseline_p - 1) * 100
improvement_r = (test_recall / baseline_r - 1) * 100
improvement_auc = (test_pr_auc / baseline_auc - 1) * 100

print(f"\n   Mejora Precision: {improvement_p:+.0f}%")
print(f"   Cambio Recall:    {improvement_r:+.0f}%")
print(f"   Mejora PR-AUC:    {improvement_auc:+.0f}%")

# 12. GUARDAR MODELO COMPACTO
log_progress("\nGuardando modelo...")

# Información mínima para Streamlit
model_artifact = {
    "model": calibrated_model,
    "scaler": scaler,
    "selector": selector,
    "selected_features": selected_features,
    "optimal_threshold": float(optimal_threshold),
    "feature_cols": feature_cols,  # Todas las features originales
    
    "performance": {
        "precision": float(test_precision),
        "recall": float(test_recall),
        "f1": float(test_f1),
        "pr_auc": float(test_pr_auc),
        "roc_auc": float(test_roc_auc)
    },
    
    "config": {
        "n_features": len(selected_features),
        "created_at": datetime.now().isoformat()
    }
}

model_path = os.path.join(SAVE_DIR, "lgbm_codespaces_optimized.joblib")
joblib.dump(model_artifact, model_path, compress=3)  # Comprimir para ahorrar espacio

# Guardar curva PR
pr_curve_df = pd.DataFrame({
    "precision": precisions[:-1],
    "recall": recalls[:-1],
    "threshold": thresholds
})
pr_path = os.path.join(RESULTS_DIR, "pr_curve.csv")
pr_curve_df.to_csv(pr_path, index=False)

# Resumen
total_time = (time.time() - start_time) / 60

print(f"\n{'='*70}")
print(f"✅ PIPELINE COMPLETADO")
print(f"{'='*70}")
print(f"⏱️  Tiempo: {total_time:.1f} minutos")
print(f"💾 Modelo guardado: {model_path}")
print(f"📈 Mejora precision: {improvement_p:+.0f}%")
print(f"🎯 Objetivo precision {TARGET_PRECISION:.0%}: {'✅ LOGRADO' if test_precision >= TARGET_PRECISION else '⚠️ No alcanzado'}")
print(f"🎯 Objetivo recall {TARGET_RECALL:.0%}: {'✅ LOGRADO' if test_recall >= TARGET_RECALL else '⚠️ No alcanzado'}")

print(f"\n🚀 PARA STREAMLIT:")
print(f"   - Modelo calibrado y optimizado")
print(f"   - Threshold: {optimal_threshold:.4f}")
print(f"   - Features: {len(selected_features)}")
print(f"   - Tamaño archivo: ~{os.path.getsize(model_path)/1024**2:.1f} MB")

🚀 PIPELINE OPTIMIZADO PARA CODESPACES
🎯 Objetivo: P≥10%, R≥70%
💾 Modo: Eficiente en memoria, sin SMOTE
[06:18:16] Iniciando pipeline...
[06:18:16] Cargando dataset con tipos optimizados...
[06:18:42] Dataset: 3,495,160 registros, 16 columnas
[06:18:43] Memoria usada: 620.5 MB
[06:18:44] Train: 2,604,597 | Val: 37,938 | Test: 447,612
[06:18:44] Accidentes - Train: 2.00%
[06:18:44] Feature engineering eficiente...
[06:18:44] Creando features de cluster (modo eficiente)...
[06:18:44]   Usando muestra de 200,000 registros para estadísticas
[06:18:45] ✅ Features de cluster creadas: 14
[06:18:45] Creando features de cluster (modo eficiente)...
[06:18:45]   Usando muestra de 200,000 registros para estadísticas
[06:18:46] ✅ Features de cluster creadas: 14
[06:18:46] Creando features de cluster (modo eficiente)...
[06:18:46]   Usando muestra de 200,000 registros para estadísticas
[06:18:46] ✅ Features de cluster creadas: 14
[06:18:46] Total features: 38
[06:18:48] Aplicando undersampling inteli